# 03 — Analyse du Chunking
**RQ2** : Quel impact ont chunk_size et chunk_overlap sur Faithfulness et Answer Relevancy ?

Grid search sur chunk_size × overlap. Ré-ingestion nécessaire pour chaque configuration.

In [ ]:
import sys, os, pandas as pd, numpy as np, itertools, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../src"))

from config import load_config
from llm_chain import generate_answer
from retriever import retrieve_documents
from evaluation_judge import create_judge
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

from ingest import create_vectorstore
from notebooks.lib.reporter import load_benchmark
from notebooks.lib.plotter import heatmap, boxplot
from experiments.registry import ExperimentLog

In [ ]:
benchmark = load_benchmark()
cfg = load_config()
judge = create_judge(cfg.evaluation)

CHUNK_SIZES = [500, 1000, 1500]
CHUNK_OVERLAPS = [0, 100, 200]

log = ExperimentLog(name="chunking_analysis")
log.set_params(chunk_sizes=CHUNK_SIZES, chunk_overlaps=CHUNK_OVERLAPS)

for size, overlap in itertools.product(CHUNK_SIZES, CHUNK_OVERLAPS):
    print(f"\n=== Chunk size={size}, overlap={overlap} ===")
    # Ré-ingestion avec nouveaux paramètres
    import ingest as ing
    ing.CHUNK_SIZE = size
    ing.CHUNK_OVERLAP = overlap
    try:
        ing.create_vectorstore()
    except Exception as e:
        print(f"Ingestion failed: {e}")
        continue

    faithfulness_scores = []
    relevancy_scores = []

    for idx, row in benchmark.iterrows():
        q_id = row['ID']
        question = row['Question']
        expected = row['Ground_Truth']

        docs, scores = retrieve_documents(question, cfg.retrieval)
        if not docs:
            continue

        answer = generate_answer(question, docs, cfg.llm)
        contexts = [d.page_content for d in docs]

        tc = LLMTestCase(
            input=question, actual_output=answer,
            expected_output=expected, retrieval_context=contexts,
        )

        fm = FaithfulnessMetric(threshold=0.75, model=judge, include_reason=True)
        ar = AnswerRelevancyMetric(threshold=0.75, model=judge, include_reason=True)

        try:
            fm.measure(tc); f_score = round(fm.score, 4)
        except: f_score = 0.0
        try:
            ar.measure(tc); a_score = round(ar.score, 4)
        except: a_score = 0.0

        log.record(ID=q_id, chunk_size=size, chunk_overlap=overlap,
                   Faithfulness=f_score, Answer_Relevancy=a_score)
        faithfulness_scores.append(f_score)
        relevancy_scores.append(a_score)

    print(f"  Faithfulness moyen: {np.mean(faithfulness_scores):.3f}")
    print(f"  Relevancy moyen: {np.mean(relevancy_scores):.3f}")

In [ ]:
# Heatmap des résultats
df = pd.DataFrame(log.results)
pivot = df.groupby(['chunk_size', 'chunk_overlap'])['Faithfulness'].mean().round(4).unstack()
print("=== Faithfulness moyenne par config chunking ===")
print(pivot)

heatmap(pivot,
        title="Faithfulness selon chunk_size et overlap",
        filename='chunking_faithfulness_heatmap.png')

log.save_all()
log.append_to_global_log()